In [1]:
from pathlib import Path
import atoti as tt
from atoti_jdbc import JdbcLoad, UserContentStorageConfig

Welcome to Atoti 0.9.9!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


In [2]:
# ===============================================
# 1️⃣ Thông tin kết nối PostgreSQL
# ===============================================
POSTGRES = {
    "host": "localhost",
    "port": 5433,
    "db": "cars",
    "user": "admin",
    "password": "admin123",
}

# Chuỗi kết nối JDBC — phải có prefix "jdbc:postgresql://"
postgres_url = (
    f"jdbc:postgresql://{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['db']}?"
    f"user={POSTGRES['user']}&password={POSTGRES['password']}"
)


In [3]:
# ===============================================
# 2️⃣ Cấu hình session Atoti để load JDBC driver
# ===============================================
# user_content_storage_config = UserContentStorageConfig(
#     postgres_url,
#     driver="org.postgresql.Driver",  # PostgreSQL JDBC driver class
# )

session_config = tt.SessionConfig(
    extra_jars=list(Path("odbc_jdbc_drivers").glob("*.jar")),  # Nạp driver .jar
    user_content_storage=Path("atoti_content"),  # Local path để lưu session
)

In [4]:
# ===============================================
# 3️⃣ Tạo session Atoti
# ===============================================
session = tt.Session.start()

# ===============================================
# 4️⃣ Định nghĩa truy vấn JDBC
# ===============================================
query = "SELECT * FROM marts.fact_car_listing"

jdbc_load = JdbcLoad(
    query=query,
    url=postgres_url,
    driver="org.postgresql.Driver",
)

In [5]:
# ===============================================
# 5️⃣ Suy luận kiểu dữ liệu và tạo bảng
# ===============================================
data_types = session.tables.infer_data_types(jdbc_load)
print("Inferred data types:", data_types)

table = session.create_table(
    "fact_car_listing",
    data_types=data_types,
    keys={"id"},
)

Inferred data types: {'id': 'long', 'price': 'long', 'mileage': 'long', 'car_general_id': 'long', 'car_details_id': 'long', 'car_specs_id': 'long', 'date_id': 'LocalDate'}


In [6]:

# ===============================================
# 6️⃣ Nạp dữ liệu trực tiếp từ PostgreSQL
# ===============================================
table.load(jdbc_load)
print("✅ Loaded data from PostgreSQL directly into Atoti!")
table.head().sort_index()

✅ Loaded data from PostgreSQL directly into Atoti!


,price,mileage,car_general_id,car_details_id,car_specs_id,date_id
id,,,,,,
274,218000000,298000,114,33,19,2025-09-18
14102,565000000,90999,433,47,8,2025-09-17
16508,358000000,234567,173,41,2,2025-09-17
19801,465000000,130000,114,49,12,2025-09-19
21504,355000000,130000,114,45,34,2025-09-13


In [7]:
# ===============================================
# 7️⃣ Tạo cube và mở UI
# ===============================================
cube = session.create_cube(table)


In [8]:
# Aliasing the hierarchies property to a shorter variable name because we will use it a lot.
h = cube.hierarchies
h

{('fact_car_listing', 'date_id'): <atoti.hierarchy.Hierarchy object at 0x11a366710>, ('fact_car_listing', 'id'): <atoti.hierarchy.Hierarchy object at 0x11a3664a0>}

In [9]:
l = cube.levels
l

{('fact_car_listing', 'date_id', 'date_id'): <atoti.level.Level object at 0x11a293040>, ('fact_car_listing', 'id', 'id'): <atoti.level.Level object at 0x11a2927a0>}

In [10]:
m = cube.measures
m

{'price.MEAN': <atoti.measure.Measure object at 0x11a293310>, 'contributors.COUNT': <atoti.measure.Measure object at 0x11a292da0>, 'car_general_id.SUM': <atoti.measure.Measure object at 0x11a315f90>, 'car_specs_id.SUM': <atoti.measure.Measure object at 0x11a315600>, 'car_details_id.SUM': <atoti.measure.Measure object at 0x11a3175b0>, 'car_general_id.MEAN': <atoti.measure.Measure object at 0x11a317070>, 'mileage.SUM': <atoti.measure.Measure object at 0x11a316350>, 'price.SUM': <atoti.measure.Measure object at 0x11a317880>, 'mileage.MEAN': <atoti.measure.Measure object at 0x11a316170>, 'car_details_id.MEAN': <atoti.measure.Measure object at 0x11a3154b0>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x11a316260>, 'car_specs_id.MEAN': <atoti.measure.Measure object at 0x11a3162c0>}

In [11]:
cube.query(m["price.MEAN"])

,price.MEAN
0,"1,015,435,277.69"


In [ ]:
cube.query(m["price.MEAN"], levels=[l["date_id"]])

,price.MEAN
date_id,
2025-06-10,"580,555,555.56"
2025-06-11,"774,000,000.00"
2025-06-12,"1,141,500,000.00"
2025-06-13,"1,407,555,555.56"
2025-06-14,"1,456,900,000.00"
...,...
2025-09-15,"1,085,664,136.62"
2025-09-16,"975,122,406.64"
2025-09-17,"1,123,204,819.28"


In [13]:
cube.query(
    m["price.MEAN"],
    filter=l["date_id"] == "2025-06-10",
)

,price.MEAN
0,"580,555,555.56"


In [15]:
print(session.url)

http://localhost:52725
